# Cài đặt thư viện cần thiết

In [1]:
!pip install -q pyvi emoji transformers scikit-learn openpyxl accelerate

import os
import json
import re
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_scheduler
from sklearn.metrics import f1_score, classification_report
from pyvi.ViTokenizer import tokenize
import emoji
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

os.makedirs("/kaggle/working/saved_models", exist_ok=True)
os.makedirs("/kaggle/working/reports", exist_ok=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 60.0 MB/s eta 0:00:00
Device: cuda


# Khai báo thư viện và kiểm tra GPU

In [2]:
import subprocess

REPO_URL = "https://github.com/ricardo-tran/ViGoEmotions.git"
REPO_DIR = "/kaggle/working/ViGoEmotions_Original"

if not os.path.exists(REPO_DIR):
    print("Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print("Repo already exists")

DOCS_PATH = os.path.join(REPO_DIR, "model", "docs")
CORPUS_PATH = os.path.join(REPO_DIR, "corpus")

# Load dictionaries
with open(os.path.join(DOCS_PATH, "patterns.json"), encoding="utf-8") as f:
    pattern_dict = json.load(f)

with open(os.path.join(DOCS_PATH, "emojis.json"), encoding="utf-8") as f:
    emoji_dict = json.load(f)

teen_dict = {}
with open(os.path.join(DOCS_PATH, "teencode4.txt"), encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line and "\t" in line:
            old, new = line.split("\t", 1)
            teen_dict[old] = new

print("✅ Dictionaries loaded")

# Load dataset
excel_path = os.path.join(CORPUS_PATH, "dataset_V1.xlsx")
excel_file = pd.ExcelFile(excel_path)

if "train" in excel_file.sheet_names:
    train_df = pd.read_excel(excel_file, sheet_name="train")
    val_df   = pd.read_excel(excel_file, sheet_name="val")
    test_df  = pd.read_excel(excel_file, sheet_name="test")
else:
    df = pd.read_excel(excel_file, sheet_name="Sheet1")
    train_df = df[df["set"] == "train"].copy()
    val_df   = df[df["set"] == "val"].copy()
    test_df  = df[df["set"] == "test"].copy()

print(f"Train: {train_df.shape} | Val: {val_df.shape} | Test: {test_df.shape}")

Cloning repo...


Cloning into '/kaggle/working/ViGoEmotions_Original'...


✅ Dictionaries loaded
Train: (16531, 3) | Val: (2066, 3) | Test: (2067, 3)


# Cấu hình đường dẫn và Tải Dữ liệu (Dictionaries & Dataset) 

In [3]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()

    # 1. Normalize patterns
    for pattern, replacement in pattern_dict.items():
        text = re.sub(pattern, replacement, text)

    # 2. Remove duplicate alphabet characters
    result, prev = [], None
    for char in text:
        if char.isalpha() and char == prev:
            continue
        prev = char
        result.append(char)
    text = "".join(result)

    # 3. Remove duplicate emojis
    result, prev_emoji = [], None
    for char in text:
        if char in emoji.EMOJI_DATA:
            if char == prev_emoji:
                continue
            prev_emoji = char
        else:
            prev_emoji = None
        result.append(char)
    text = "".join(result)

    # 4. Replace teencode
    for old, new in teen_dict.items():
        text = re.sub(rf"\b{re.escape(old)}\b", new, text)

    # 5. Replace emojis
    for emj, rep in emoji_dict.items():
        text = text.replace(emj, f" {rep} ")

    # 6. Format punctuation & whitespace
    text = re.sub(r"(?<![.,!?;:])\n", ". ", text)
    text = re.sub(r"\n([.,!?;:])?", r" \1", text)
    text = re.sub(r"([.,!?;:])", r" \1 ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("Applying S2 preprocessing...")
for df in [train_df, val_df, test_df]:
    df["text"] = df["text"].astype(str).apply(clean_text)

print("✅ Preprocessing done")
print(train_df["text"].iloc[0])

Applying S2 preprocessing...
✅ Preprocessing done
xem mà ngẫm lại cuộc đời bản thân ta đã trải qua nhiều thứ ta rồi cũng sẽ lớn kí ước sẽ còn mãi trong lòng


In [4]:
!find /kaggle/working/ViGoEmotions_Original -maxdepth 3 -type f

/kaggle/working/ViGoEmotions_Original/annotation/groq_annotator_llama_3_70b_public.ipynb
/kaggle/working/ViGoEmotions_Original/annotation/llm_guideline_official.md
/kaggle/working/ViGoEmotions_Original/annotation/google_annotator_gemma_3_public.ipynb
/kaggle/working/ViGoEmotions_Original/annotation/google_annotator_gemini_2_0f_public.ipynb
/kaggle/working/ViGoEmotions_Original/README.md
/kaggle/working/ViGoEmotions_Original/corpus/train.csv
/kaggle/working/ViGoEmotions_Original/corpus/val.csv
/kaggle/working/ViGoEmotions_Original/corpus/test.csv
/kaggle/working/ViGoEmotions_Original/corpus/label_dict.json
/kaggle/working/ViGoEmotions_Original/corpus/dataset_V1.xlsx
/kaggle/working/ViGoEmotions_Original/.git/HEAD
/kaggle/working/ViGoEmotions_Original/.git/packed-refs
/kaggle/working/ViGoEmotions_Original/.git/info/exclude
/kaggle/working/ViGoEmotions_Original/.git/logs/HEAD
/kaggle/working/ViGoEmotions_Original/.git/config
/kaggle/working/ViGoEmotions_Original/.git/description
/kaggle/w

# Tiền xử lý văn bản (S2 Preprocessing)

In [5]:
with open(os.path.join(DOCS_PATH, "label_dict.json"), encoding="utf-8") as f:
    label_dict = json.load(f)

label_to_idx = {label: int(idx) for idx, label in label_dict.items()}
print("Number of labels:", len(label_dict))

def encode_labels(label_str, label_dict):
    labels = str(label_str).replace("[", "").replace("]", "").replace("'", "").replace('"', "").split(",")
    labels = [x.strip() for x in labels if x.strip()]
    vec = np.zeros(len(label_dict), dtype=np.float32)

    if labels and labels[0].isnumeric():
        labels = [int(x) for x in labels]
        for idx in label_dict.values():
            if idx in labels:
                vec[idx] = 1.0
    else:
        for lab, idx in label_dict.items():
            if lab in labels:
                vec[idx] = 1.0
    return vec

train_texts  = train_df["text"].tolist()
train_labels = [encode_labels(x, label_to_idx) for x in train_df["labels"]]
val_texts    = val_df["text"].tolist()
val_labels   = [encode_labels(x, label_to_idx) for x in val_df["labels"]]
test_texts   = test_df["text"].tolist()
test_labels  = [encode_labels(x, label_to_idx) for x in test_df["labels"]]

print("✅ Labels encoded")
print("Example label:", train_labels[0])

Number of labels: 28
✅ Labels encoded
Example label: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0.]


# Load Labels và Mã hóa One-hot (Label Encoding)

In [6]:
# ===== CHỌN MODEL Ở ĐÂY =====
model_type = "visobert"          # có thể đổi: phobert, xlm-r, mbert, bartpho, cafebert...
model_name = "uitnlp/visobert"
max_len = 200
BATCH_SIZE = 16                  # giảm xuống 16 để tránh treo / OOM
# ============================

tokenizer = AutoTokenizer.from_pretrained(model_name)

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = torch.tensor(labels, dtype=torch.float32)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        # S2: một số model dùng pyvi, một số không
        # Với visobert / phobert / xlm-r thường KHÔNG cần tokenize() của pyvi
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt",
            return_attention_mask=True,
        )
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "targets": self.labels[idx],
            "text": text,
        }

train_dataset = SentimentDataset(train_texts, train_labels, tokenizer, max_len)
val_dataset   = SentimentDataset(val_texts, val_labels, tokenizer, max_len)
test_dataset  = SentimentDataset(test_texts, test_labels, tokenizer, max_len)

# QUAN TRỌNG: num_workers=0 để tránh treo trên Kaggle
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("✅ DataLoader ready")
print("Train batches:", len(train_loader))

config.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/471k [00:00<?, ?B/s]

✅ DataLoader ready
Train batches: 1034


/tmp/ipykernel_22/3205300498.py:13: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  self.labels = torch.tensor(labels, dtype=torch.float32)


# Khởi tạo Tokenizer và DataLoader

In [7]:
class ModelSentimentClassifier(nn.Module):
    def __init__(self, n_classes, model_name, model_type):
        super().__init__()
        self.model_type = model_type
        config = AutoConfig.from_pretrained(
            model_name,
            hidden_dropout_prob=0.1,
            attention_probs_dropout_prob=0.1
        )
        self.backbone = AutoModel.from_pretrained(model_name, config=config)
        self.drop = nn.Dropout(0.2)
        self.fc = nn.Linear(self.backbone.config.hidden_size, n_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        if "bartpho" in self.model_type:
            pooled = outputs.last_hidden_state[:, 0, :]
        else:
            pooled = outputs.pooler_output if outputs.pooler_output is not None else outputs.last_hidden_state[:, 0, :]
        x = self.drop(pooled)
        return {"logits": self.fc(x)}

model = ModelSentimentClassifier(
    n_classes=len(label_dict),
    model_name=model_name,
    model_type=model_type
).to(device)

print("✅ Model loaded")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

pytorch_model.bin:   0%|          | 0.00/390M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/390M [00:00<?, ?B/s]

✅ Model loaded
Parameters: 97,587,484


# Cấu hình

In [8]:
EPOCHS = 12
optimizer = AdamW(model.parameters(), lr=5e-5)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=len(train_loader),
    num_training_steps=len(train_loader) * EPOCHS
)

# pos_weight chống mất cân bằng
label_counts = np.sum(train_labels, axis=0)
pos_weight = torch.tensor(
    [(len(train_labels) - c) / max(c, 1) for c in label_counts],
    dtype=torch.float32
).to(device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

def run_epoch(model, loader, is_train=True):
    model.train() if is_train else model.eval()
    losses, all_y, all_p = [], [], []

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for batch in tqdm(loader, leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            targets = batch["targets"].to(device)

            if is_train:
                optimizer.zero_grad()

            logits = model(input_ids, attention_mask)["logits"]
            loss = loss_fn(logits, targets)
            losses.append(loss.item())

            if is_train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                lr_scheduler.step()

            preds = (torch.sigmoid(logits) >= 0.5).int()
            all_y.append(targets.cpu().numpy())
            all_p.append(preds.cpu().numpy())

    y = np.vstack(all_y)
    p = np.vstack(all_p)
    f1 = f1_score(y, p, average="macro", zero_division=0)
    return np.mean(losses), f1

best_f1 = 0.0
history = {"train_loss": [], "train_f1": [], "val_loss": [], "val_f1": []}

print("Start training...")
for epoch in range(1, EPOCHS + 1):
    print(f"\n===== Epoch {epoch}/{EPOCHS} =====")
    train_loss, train_f1 = run_epoch(model, train_loader, is_train=True)
    val_loss, val_f1 = run_epoch(model, val_loader, is_train=False)

    history["train_loss"].append(train_loss)
    history["train_f1"].append(train_f1)
    history["val_loss"].append(val_loss)
    history["val_f1"].append(val_f1)

    print(f"Train Loss: {train_loss:.4f} | Train Macro-F1: {train_f1:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Macro-F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), f"/kaggle/working/saved_models/{model_type}_best.pth")
        print(f"⭐ Saved best model (Val F1 = {best_f1:.4f})")

print("\n✅ Training finished!")

Start training...

===== Epoch 1/12 =====



100%|██████████| 1034/1034 [10:09<00:00,  2.16it/s]


Train Loss: 1.0288 | Train Macro-F1: 0.2346
Val   Loss: 0.8104 | Val   Macro-F1: 0.3422
⭐ Saved best model (Val F1 = 0.3422)

===== Epoch 2/12 =====


Train Loss: 0.7192 | Train Macro-F1: 0.3990
Val   Loss: 0.7224 | Val   Macro-F1: 0.4052
⭐ Saved best model (Val F1 = 0.4052)

===== Epoch 3/12 =====


Train Loss: 0.5069 | Train Macro-F1: 0.5129
Val   Loss: 0.7613 | Val   Macro-F1: 0.4663
⭐ Saved best model (Val F1 = 0.4663)

===== Epoch 4/12 =====


...

Train Loss: 0.1211 | Train Macro-F1: 0.8671
Val   Loss: 1.3236 | Val   Macro-F1: 0.5445

===== Epoch 9/12 =====


Train Loss: 0.0877 | Train Macro-F1: 0.9083
Val   Loss: 1.4490 | Val   Macro-F1: 0.5487
⭐ Saved best model (Val F1 = 0.5487)

===== Epoch 10/12 =====


Train Loss: 0.0648 | Train Macro-F1: 0.9412
Val   Loss: 1.5603 | Val   Macro-F1: 0.5550
⭐ Saved best model (Val F1 = 0.5550)

===== Epoch 11/12 =====


Train Loss: 0.0466 | Train Macro-F1: 0.9615
Val   Loss: 1.6554 | Val   Macro-F1: 0.5586
⭐ Saved best model (Val F1 = 0.5586)

===== Epoch 12/12 =====


Train Loss: 0.0363 | Train Macro-F1: 0.9751
Val   Loss: 1.6829 | Val   Macro-F1: 0.5561

✅ Training finished!


# Thiết lập Hàm Huấn luyện & Đánh giá

In [9]:
# Load best model
model.load_state_dict(torch.load(f"/kaggle/working/saved_models/{model_type}_best.pth"))
model.eval()

all_targets, all_preds = [], []

with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets = batch["targets"].to(device)

        logits = model(input_ids, attention_mask)["logits"]
        preds = (torch.sigmoid(logits) >= 0.5).int()

        all_targets.append(targets.cpu().numpy())
        all_preds.append(preds.cpu().numpy())

y_true = np.vstack(all_targets)
y_pred = np.vstack(all_preds)

macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)

print("\n========== TEST RESULTS ==========")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Micro F1: {micro_f1:.4f}")
print("\nClassification Report:")
print(classification_report(
    y_true, y_pred,
    target_names=list(label_dict.values()),
    zero_division=0
))

100%|██████████| 130/130 [00:24<00:00,  5.31it/s]


========== TEST RESULTS ==========
Macro F1: 0.5688
Micro F1: 0.5871

Classification Report:
                precision    recall  f1-score   support

     amusement       0.70      0.79      0.75       374
    excitement       0.52      0.38      0.44        98
           joy       0.50      0.55      0.52       204
          love       0.59      0.73      0.66       143
        desire       0.42      0.54      0.47        80
      optimism       0.59      0.66      0.62       142
        caring       0.60      0.65      0.62       150
         pride       0.60      0.59      0.60        86
    admiration       0.58      0.59      0.59       101
     gratitude       0.83      0.88      0.86       108
        relief       0.47      0.58      0.52        60
      approval       0.51      0.58      0.54       115
   realization       0.41      0.43      0.42        95
      surprise       0.53      0.51      0.52        85
     curiosity       0.57      0.66      0.61       100
     conf

# train loop